In [ ]:
# Read dataset
import pandas as pd

df = pd.read_csv("../data/raw/Paid_Parking_Occupancy_Mar28-Mar29_2026.csv")

print(df.shape)
print(df.columns)
df.head()

(997825, 12)
Index(['OccupancyDateTime', 'PaidOccupancy', 'BlockfaceName', 'SideOfStreet',
       'SourceElementKey', 'ParkingTimeLimitCategory', 'ParkingSpaceCount',
       'PaidParkingArea', 'PaidParkingSubArea', 'PaidParkingRate',
       'ParkingCategory', 'Location'],
      dtype='object')


,OccupancyDateTime,PaidOccupancy,BlockfaceName,SideOfStreet,SourceElementKey,ParkingTimeLimitCategory,ParkingSpaceCount,PaidParkingArea,PaidParkingSubArea,PaidParkingRate,ParkingCategory,Location
0,2026 Mar 28 08:00:00 AM,1,5TH AVE BETWEEN VIRGINIA ST AND LENORA ST,SW,"7,257",120.0,11,Belltown,South,NaN,Paid Parking,POINT (-122.33988424 47.61405613)
1,2026 Mar 28 08:00:00 AM,1,11TH AVE BETWEEN E PINE ST AND E OLIVE ST,E,"24,410",240.0,5,Capitol Hill,South,NaN,Paid Parking,POINT (-122.31805976 47.61586246)
2,2026 Mar 28 08:00:00 AM,1,MERCER SR ST BETWEEN WESTLAKE AVE N AND TERRY ...,S,"11,778",600.0,4,South Lake Union,South,NaN,Paid Parking,POINT (-122.33783456 47.62446799)
3,2026 Mar 28 08:00:00 AM,1,S JACKSON ST BETWEEN 6TH AVE S AND MAYNARD AVE S,S,"88,546",120.0,1,Chinatown/ID,NaN,NaN,Paid Parking,POINT (-122.32573848 47.59910134)
4,2026 Mar 28 08:00:00 AM,1,REPUBLICAN ST BETWEEN TERRY AVE N AND BOREN AVE N,N,"35,197",120.0,8,South Lake Union,South,NaN,Paid Parking,POINT (-122.33651591 47.62326784)


In [ ]:
# Select relevant columns
df = df[[
    "OccupancyDateTime",
    "BlockfaceName",
    "PaidOccupancy",
    "ParkingSpaceCount",
    "PaidParkingArea",
    "ParkingTimeLimitCategory"
]].copy()

In [ ]:
# Data type cleaning
df["OccupancyDateTime"] = pd.to_datetime(df["OccupancyDateTime"], errors="coerce")
df["PaidOccupancy"] = pd.to_numeric(df["PaidOccupancy"], errors="coerce")
df["ParkingSpaceCount"] = pd.to_numeric(df["ParkingSpaceCount"], errors="coerce")

In [ ]:
# Drop unusable rows
df = df.dropna(subset=["BlockfaceName", "PaidOccupancy"])
df.head()

,OccupancyDateTime,BlockfaceName,PaidOccupancy,ParkingSpaceCount,PaidParkingArea,ParkingTimeLimitCategory
0,2026-03-28 08:00:00,5TH AVE BETWEEN VIRGINIA ST AND LENORA ST,1,11,Belltown,120.0
1,2026-03-28 08:00:00,11TH AVE BETWEEN E PINE ST AND E OLIVE ST,1,5,Capitol Hill,240.0
2,2026-03-28 08:00:00,MERCER SR ST BETWEEN WESTLAKE AVE N AND TERRY ...,1,4,South Lake Union,600.0
3,2026-03-28 08:00:00,S JACKSON ST BETWEEN 6TH AVE S AND MAYNARD AVE S,1,1,Chinatown/ID,120.0
4,2026-03-28 08:00:00,REPUBLICAN ST BETWEEN TERRY AVE N AND BOREN AVE N,1,8,South Lake Union,120.0


In [ ]:
# Check value count of parking areas 
df["PaidParkingArea"].value_counts()

PaidParkingArea
Belltown               164880
South Lake Union       114600
Uptown                  97783
First Hill              92400
Commercial Core         89533
Pike-Pine               71976
Denny Triangle          61920
University District     56473
Chinatown/ID            47990
Pioneer Square          43743
Ballard                 43200
Capitol Hill            31080
Green Lake              22320
Uptown Triangle         12000
15th Avenue E           11520
12th Avenue              8640
Roosevelt                7920
Columbia City            7200
Fremont                  6167
Cherry Hill              4320
Ballard Locks            2160
Name: count, dtype: int64

In [9]:
# Filter to University District for ease of work while still exposing coverage and measurement limitations
df = df[df["PaidParkingArea"] == "University District"]

In [11]:
# Aggregate
agg = (
    df.groupby("BlockfaceName")
    .agg(
        avg_paid_occupancy=("PaidOccupancy", "mean"),
        avg_spaces=("ParkingSpaceCount", "mean"),
        observations=("PaidOccupancy", "count")
    )
    .reset_index()
)

In [12]:
# Identify coverage (data density)
import numpy as np

agg["coverage_level"] = np.where(
    agg["observations"] < 500,
    "Low Coverage",
    "High Coverage"
)

In [13]:
# Occupancy ratio
agg["occupancy_ratio"] = agg["avg_paid_occupancy"] / agg["avg_spaces"]

In [14]:
# Sort and reduce for visualization
agg = agg.sort_values(by="avg_paid_occupancy", ascending=False).reset_index(drop=True)

final_df = agg.head(50).copy()

In [15]:
# Format
final_df["avg_paid_occupancy"] = final_df["avg_paid_occupancy"].round(2)
final_df["avg_spaces"] = final_df["avg_spaces"].round(2)
final_df["occupancy_ratio"] = final_df["occupancy_ratio"].round(2)

final_df.head()

,BlockfaceName,avg_paid_occupancy,avg_spaces,observations,coverage_level,occupancy_ratio
0,15TH AVE NE BETWEEN NE 45TH ST AND NE 47TH ST,12.92,12.0,1200,High Coverage,1.08
1,15TH AVE NE BETWEEN NE 47TH ST AND NE 50TH ST,11.53,20.0,1200,High Coverage,0.58
2,BROOKLYN AVE NE BETWEEN NE 41ST ST AND NE 42ND ST,9.78,17.0,1440,High Coverage,0.58
3,12TH AVE NE BETWEEN NE 41ST ST AND NE 42ND ST,9.40,17.0,720,High Coverage,0.55
4,NE BOAT ST BETWEEN NE BOAT WR ST AND BROOKLYN ...,8.73,30.5,1440,High Coverage,0.29


In [16]:
# Save output csv
final_df.to_csv("../data/processed/parking_udistrict_analysis.csv", index=False)